# Bayaan Class A — Phoneme Recognizer Training (MVP)

Trains a frozen-backbone wav2vec2-CTC head to output the Quranic Phonemizer's native 69-symbol phoneme inventory, on Al-Fatihah (surah 1) + Al-Bayyinah (surah 98) clips from `hetchyy/everyayah-phonemes`.

MVP scope only — proves the pipeline (recognizer -> CTC decode -> PER), not a production-scale model. See `/home/jade/.claude/plans/serialized-baking-cat.md` in the Bayaan repo for the full plan.

**Optional:** add an `HF_TOKEN` Kaggle Secret (Settings -> Add-ons -> Secrets) to avoid HF rate limits. Not required — the model and dataset are both public.


In [ ]:
!pip install -q transformers soundfile editdistance pyarrow huggingface_hub


In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("hugging face")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as e:
    print(f"No HF_TOKEN secret found ({type(e).__name__}: {e}) — proceeding without it (public resources, just slower/rate-limited)")


In [ ]:
import io
import json
import random
import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU attached — check Settings > Accelerator, and that your Kaggle account is phone-verified (required for GPU quota).")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Verified 2026-06-20 by phonemizing all 6,236 verses of the Quran via
# `quranic_phonemizer` and collecting `get_mapping().phoneme_sequence` —
# do not trust any other quoted vocab size (e.g. "71") without re-deriving this.
PHONEMES = sorted([
    "Q", "a", "a:", "aˤ", "aˤ:", "b", "bb", "d", "dd", "dˤ", "dˤdˤ",
    "f", "ff", "h", "hh", "i", "i:", "j", "jj", "j̃", "k", "kk", "l", "ll", "lˤlˤ",
    "m", "m̃", "n", "q", "qq", "r", "rr", "rˤ", "rˤrˤ", "s", "ss", "sˤ", "sˤsˤ",
    "t", "tt", "tˤ", "tˤtˤ", "u", "u:", "w", "ww", "w̃", "x", "xx", "z", "zz",
    "ð", "ðð", "ðˤ", "ðˤðˤ", "ñ", "ħ", "ħħ",
    "ŋ", "ɣ", "ʃ", "ʃʃ", "ʒ", "ʒʒ", "ʔ", "ʕ", "ʕʕ",
    "θ", "θθ",
])
assert len(PHONEMES) == 69, f"expected 69 phonemes, got {len(PHONEMES)} — re-verify against the Phonemizer"

BLANK_ID = 0
PHON_TO_ID = {p: i + 1 for i, p in enumerate(PHONEMES)}  # 0 reserved for CTC blank
ID_TO_PHON = {i: p for p, i in PHON_TO_ID.items()}
VOCAB_SIZE = len(PHONEMES) + 1  # 70
print("vocab size (incl. blank):", VOCAB_SIZE)


## Manifest — scan parquet shards directly on Kaggle (no `/filter` API, no embedding)

The first training run (2026-06-20, val_PER 0.413) pre-fetched clips via HF's `/filter` endpoint
from the orchestrating machine and embedded the result as a literal JSON blob, because live
`/filter` calls are unreliable from Kaggle's network (100% retry exhaustion). That endpoint also
only indexes the **first 5GB of the 207k-row train split**, which silently capped training data
at 4 reciters — well short of the 15-20 originally planned. The embedded URLs also expired after
~1 hour, not days, forcing a fetch-immediately-before-push workflow.

This cell sidesteps all three problems by reading the dataset's parquet shards directly
(`data/train-NNNNN-of-00124.parquet`, `data/dev-NNNNN-of-00006.parquet`) — a different, more
reliable code path than `/filter`, and one with no 5GB index limit. Audio is embedded as raw
bytes in each shard's `audio` struct column, so no presigned URLs (and no expiry) are involved
at all. This was bandwidth-bound when tried from the local dev sandbox (~12 min per 428MB shard);
Kaggle's network is far faster, so the scan runs here instead of being pre-fetched and embedded.

Each shard is ~one reciter's full repertoire (~4,600 rows; 124 shards / 45 reciters ≈ 2.76
shards/reciter) — stepping the shard index by 3 lands on a new reciter almost every time. A cheap
metadata-only pass (`columns=["reciter"]`, no audio) checks each candidate shard before paying for
the full audio download.


In [ ]:
from huggingface_hub import list_repo_files
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import requests

REPO = "hetchyy/everyayah-phonemes"
TARGET_VERSES = {f"1_{i}" for i in range(1, 8)} | {f"98_{i}" for i in range(1, 9)}
NEW_RECITER_TARGET = 16  # plan's originally intended 15-20 reciters for train

repo_files = list_repo_files(REPO, repo_type="dataset")
train_files = sorted(f for f in repo_files if f.startswith("data/train-"))
dev_files = sorted(f for f in repo_files if f.startswith("data/dev-"))
print(f"{len(train_files)} train shards, {len(dev_files)} dev shards")


def read_columns(shard_path: str, columns: list[str]) -> pd.DataFrame:
    # pyarrow's native hf:// reader supports column-chunk pruning (skips audio bytes
    # when not requested) -- much cheaper than a full download. Falls back to a plain
    # HTTP fetch if the Kaggle image's pyarrow build lacks hf:// support.
    try:
        t = pq.read_table(f"hf://datasets/{REPO}/{shard_path}", columns=columns)
    except Exception:
        url = f"https://huggingface.co/datasets/{REPO}/resolve/main/{shard_path}"
        import io
        t = pq.read_table(io.BytesIO(requests.get(url, timeout=120).content), columns=columns)
    return t.to_pandas()


def shard_reciters(shard_path: str) -> set[str]:
    return set(read_columns(shard_path, ["reciter"])["reciter"].unique())


def matching_rows(shard_path: str) -> pd.DataFrame:
    df = read_columns(shard_path, ["audio", "reciter", "verse", "phonemes"])
    return df[df["verse"].isin(TARGET_VERSES)]


def to_clip(row) -> dict:
    waveform, sr = sf.read(io.BytesIO(row["audio"]["bytes"]), dtype="float32")
    assert sr == 16_000, f"expected 16kHz, got {sr}Hz for {row['verse']}/{row['reciter']}"
    return {"verse": row["verse"], "reciter": row["reciter"], "phonemes": row["phonemes"].split(" "), "waveform": waveform}


# dev: only 6 shards total, scan all of them fully
dev_chunks = [matching_rows(f) for f in dev_files]
dev_df = pd.concat(dev_chunks, ignore_index=True)
print(f"dev: {len(dev_df)} clips, {dev_df['reciter'].nunique()} reciters")

# train: a reciter's full recitation runs in canonical surah order across ~2.76 contiguous
# shards, so Al-Fatihah (surah 1, the start of their block) and Al-Bayyinah (surah 98, near the
# end) usually land in DIFFERENT shards for the same reciter. Reading only the single shard
# where a reciter is first spotted (the previous version of this cell) silently caught
# Al-Fatihah and dropped Al-Bayyinah for almost everyone. Fix: once a probe finds a new
# reciter, pull a WINDOW-shard span starting there -- wide enough to cover one reciter's whole
# block with margin for 2.76 not being an exact 3.
WINDOW = 4
train_chunks, reciters_seen, shard_idx = [], set(), 0
while len(reciters_seen) < NEW_RECITER_TARGET and shard_idx < len(train_files):
    new_reciters = shard_reciters(train_files[shard_idx]) - reciters_seen
    if new_reciters:
        window_chunks = [matching_rows(f) for f in train_files[shard_idx:shard_idx + WINDOW]]
        window_df = pd.concat(window_chunks, ignore_index=True)
        window_df = window_df[window_df["reciter"].isin(new_reciters)]
        if len(window_df):
            train_chunks.append(window_df)
            found = set(window_df["reciter"].unique())
            reciters_seen |= found
            print(f"shard {shard_idx}+{WINDOW}: +{len(found)} reciters ({len(window_df)} clips), total {len(reciters_seen)}")
    shard_idx += WINDOW

train_df = pd.concat(train_chunks, ignore_index=True)
print(f"train: {len(train_df)} clips, {train_df['reciter'].nunique()} reciters")

overlap = set(train_df["reciter"]) & set(dev_df["reciter"])
assert not overlap, f"train/dev reciter overlap, would leak: {overlap}"

train_manifest = [to_clip(r) for _, r in train_df.iterrows()]
dev_manifest = [to_clip(r) for _, r in dev_df.iterrows()]


## Dataset — phonemes -> ids, audio already decoded in-memory from the scan above


In [ ]:
class PhonemeClipDataset(Dataset):
    def __init__(self, manifest: list[dict]):
        self.manifest = manifest

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, i):
        item = self.manifest[i]
        ids = [PHON_TO_ID[p] for p in item["phonemes"]]
        return torch.from_numpy(item["waveform"]), torch.tensor(ids, dtype=torch.long)


def collate(batch):
    waveforms, targets = zip(*batch)
    wav_lens = torch.tensor([len(w) for w in waveforms])
    tgt_lens = torch.tensor([len(t) for t in targets])
    wav_padded = nn.utils.rnn.pad_sequence(waveforms, batch_first=True)
    tgt_concat = torch.cat(targets)  # CTCLoss wants targets concatenated, not padded
    return wav_padded, wav_lens, tgt_concat, tgt_lens


train_ds = PhonemeClipDataset(train_manifest)
dev_ds = PhonemeClipDataset(dev_manifest)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(dev_ds, batch_size=8, shuffle=False, collate_fn=collate)
print(f"train batches: {len(train_loader)}, dev batches: {len(dev_loader)}")


## Model — frozen XLSR-Arabic backbone (same one Class B already uses) + CTC head

In [ ]:
BACKBONE = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"  # matches ml/src/config.py's BACKBONE


class PhonemeRecognizer(nn.Module):
    def __init__(self, vocab_size: int = VOCAB_SIZE, freeze_backbone: bool = True):
        super().__init__()
        self.backbone = Wav2Vec2Model.from_pretrained(BACKBONE)
        if freeze_backbone:
            self.backbone.eval()
            for p in self.backbone.parameters():
                p.requires_grad = False
        self.head = nn.Linear(self.backbone.config.hidden_size, vocab_size)
        self.freeze_backbone = freeze_backbone

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        ctx = torch.no_grad() if self.freeze_backbone else torch.enable_grad()
        with ctx:
            hidden = self.backbone(waveform).last_hidden_state  # (B, T, H)
        return self.head(hidden)  # (B, T, vocab_size) — log_softmax applied at loss time

    def train(self, mode: bool = True):
        # nn.Module.train() recurses into every submodule, including a frozen backbone --
        # that silently re-enables its dropout (if the checkpoint's config has any nonzero
        # dropout prob) even though requires_grad=False keeps its weights from updating.
        # Force it back to eval so "frozen" also means "deterministic", not just "untrained".
        super().train(mode)
        if self.freeze_backbone:
            self.backbone.eval()
        return self


model = PhonemeRecognizer().to(DEVICE)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params (head only): {n_trainable:,}")


## PER (phoneme error rate) via edit distance, and the training loop

In [ ]:
import editdistance


def greedy_ctc_decode(logits: torch.Tensor) -> list[int]:
    """logits: (T, vocab_size) for one clip -> collapsed phoneme id sequence."""
    ids = logits.argmax(dim=-1).tolist()
    out, prev = [], None
    for i in ids:
        if i != prev and i != BLANK_ID:
            out.append(i)
        prev = i
    return out


@torch.no_grad()
def evaluate(model, loader) -> float:
    model.eval()
    total_edits, total_len = 0, 0
    for wav, wav_lens, tgt, tgt_lens in loader:
        wav = wav.to(DEVICE)
        logits = model(wav)  # (B, T, V)
        offset = 0
        for b in range(wav.shape[0]):
            pred = greedy_ctc_decode(logits[b])
            ref = tgt[offset:offset + tgt_lens[b]].tolist()
            offset += tgt_lens[b]
            total_edits += editdistance.eval(pred, ref)
            total_len += len(ref)
    return total_edits / max(total_len, 1)


def train(model, train_loader, dev_loader, epochs=60, lr=1e-3, patience=10):
    criterion = nn.CTCLoss(blank=BLANK_ID, zero_infinity=True)
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=lr)
    best_per, best_state, no_improve = float("inf"), None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for wav, wav_lens, tgt, tgt_lens in train_loader:
            wav, tgt = wav.to(DEVICE), tgt.to(DEVICE)
            logits = model(wav)  # (B, T, V)
            log_probs = logits.log_softmax(dim=-1).transpose(0, 1)  # (T, B, V) for CTCLoss
            # backbone may subsample time steps internally; assume uniform stride across the batch
            input_lens = torch.full((wav.shape[0],), logits.shape[1], dtype=torch.long)
            loss = criterion(log_probs, tgt, input_lens, tgt_lens)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        val_per = evaluate(model, dev_loader)
        print(f"epoch {epoch:3d} | train_loss {total_loss / len(train_loader):.3f} | val_PER {val_per:.3f}")

        if val_per < best_per:
            best_per, best_state, no_improve = val_per, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"early stop at epoch {epoch} (best val_PER {best_per:.3f})")
                break

    model.load_state_dict(best_state)
    return model, best_per

In [ ]:
model, best_per = train(model, train_loader, dev_loader)
print(f"\nFINAL best val_PER: {best_per:.3f}")

In [ ]:
checkpoint = {
    "head_state_dict": model.head.state_dict(),
    "phon_to_id": PHON_TO_ID,
    "vocab_size": VOCAB_SIZE,
    "blank_id": BLANK_ID,
    "backbone": BACKBONE,
    "best_val_per": best_per,
}
torch.save(checkpoint, "/kaggle/working/phoneme_recognizer_mvp.pt")
print("saved to /kaggle/working/phoneme_recognizer_mvp.pt")
